In [28]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%%bash
set -e

MAMBA_DIR="/content/drive/MyDrive/micromamba"
mkdir -p "$MAMBA_DIR"

curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest \
  | tar -xvj -C "$MAMBA_DIR" bin/micromamba

ls -l "$MAMBA_DIR/bin/micromamba"

In [ ]:
%%bash
set -e
MM="/content/drive/MyDrive/micromamba/bin/micromamba"
export MAMBA_ROOT_PREFIX="/content/drive/MyDrive/mamba_root"

"$MM" env remove -y -n py36_mlgraphdt || true
"$MM" create -y -n py36_mlgraphdt -c conda-forge "python=3.6.*" pip
"$MM" run -n py36_mlgraphdt python -V

In [ ]:
# list of dependencies in environment
# %%bash
# MM="/content/drive/MyDrive/micromamba/bin/micromamba"
# export MAMBA_ROOT_PREFIX="/content/drive/MyDrive/mamba_root"


# "$MM" list -n py36_mlgraphdt


In [32]:
%%bash
set -e

MM="/content/drive/MyDrive/micromamba/bin/micromamba"
export MAMBA_ROOT_PREFIX="/content/drive/MyDrive/mamba_root"

# "$MM"  install -y -n py36_mlgraphdt -c conda-forge pycuda
# "$MM" run -n py36_mlgraphdt python -m pip install --force-reinstall "ruamel.yaml==0.17.21"

# "$MM" install -y -n py36_mlgraphdt -c conda-forge rdkit
# "$MM" run -n py36_mlgraphdt python -m pip uninstall -y pytools
# "$MM" run -n py36_mlgraphdt python -m pip install "pytools==2020.4.4"

# "$MM" install -y -n py36_mlgraphdt -c conda-forge "pymatgen==2019.11.11"
# "$MM" run -n py36_mlgraphdt python -c "import pymatgen; print('pymatgen', pymatgen.__version__)"

"$MM" run -n py36_mlgraphdt python -m pip install -U graphdot
"$MM" run -n py36_mlgraphdt python -c "import graphdot; print('graphdot OK', graphdot.__version__)"

  Using cached graphdot-0.8.1-py3-none-any.whl
  Using cached numba-0.53.1-cp36-cp36m-manylinux2014_x86_64.whl (3.4 MB)
  Using cached kahypar-1.3.5-cp36-cp36m-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.2 MB)
  Using cached mendeleev-0.9.0-py3-none-any.whl (178 kB)
  Using cached treelib-1.7.1-py3-none-any.whl (19 kB)
  Using cached ase-3.22.1-py3-none-any.whl (2.2 MB)
  Using cached Pygments-2.14.0-py3-none-any.whl (1.1 MB)
  Using cached pyfiglet-0.8.post1-py2.py3-none-any.whl (865 kB)
  Using cached llvmlite-0.36.0-cp36-cp36m-manylinux2010_x86_64.whl (25.3 MB)
  Attempting uninstall: ase
    Found existing installation: ase 3.23.0
    Uninstalling ase-3.23.0:
      Successfully uninstalled ase-3.23.0
graphdot OK 0.8.1


In [34]:
%%bash
MM="/content/drive/MyDrive/micromamba/bin/micromamba"
export MAMBA_ROOT_PREFIX="/content/drive/MyDrive/mamba_root"

"$MM" run -n py36_mlgraphdt python - <<'PY'
import graphdot
print("graphdot version:", graphdot.__version__)
PY


graphdot version: 0.8.1


In [ ]:
%%bash
MM="/content/drive/MyDrive/micromamba/bin/micromamba"
export MAMBA_ROOT_PREFIX="/content/drive/MyDrive/mamba_root"

"$MM" run -n py36_mlgraphdt python - <<'PY'

import os, glob
import pandas as pd
import numpy as np
import time
from ase.io import read
from graphdot import Graph
from graphdot.graph.adjacency import AtomicAdjacency
from graphdot.model.gaussian_process import GaussianProcessRegressor
from graphdot.kernel.fix import Normalization
from graphdot.kernel.molecular import Tang2019MolecularKernel as MolecularKernel


data_path = "/content/drive/MyDrive/QE_Fe/PBE/High-pressure/300gpa_5000K/"
gpr = GaussianProcessRegressor(
    # kernel is the covariance function of the gaussian process (GP)
    kernel=Normalization(MolecularKernel()),
    alpha=1e-4, # value added to the diagonal of the kernel matrix during fitting
    optimizer=True, # default optimizer of L-BFGS-B based on scipy.optimize.minimize
    normalize_y=True, # normalize the y values so taht the means and variance is 0 and 1, repsectively. Will be reversed when predicions are returned
)

data_dir = "/content/drive/MyDrive/QE_Fe/PBE/High-pressure/300gpa_5000K/"
files = [os.path.join(data_dir, f"fe{i}.out") for i in range(1, 6)]
print("Files:", files)

atoms_list = []
for fp in files:
    if not os.path.exists(fp):
        print("Missing:", fp)
        continue
    atoms_list.extend(read(fp, index=":", format="espresso-out"))

print("Total frames read:", len(atoms_list))

graphs = [Graph.from_ase(a, adjacency=AtomicAdjacency(shape="compactbell3,2")) for a in atoms_list]
energy_gt = [a.get_potential_energy() for a in atoms_list]

data = pd.DataFrame({"Graphs": graphs, "Pot_Energy": energy_gt})
print(data.head())


ntsteps = len(atoms_list)
target = 'Pot_Energy' # Learning target from the dataframe
N_test = ntsteps//3 # Number of testing steps
N_train = ntsteps # For when I do not care about testing and want the best possible model
# N_train = 2*N_test # Number of training steps
print(N_train)
print(N_test)
np.random.seed(0)
# select train and test data
train_sel = np.random.choice(ntsteps, N_train, replace=False)
test_sel = np.random.choice(ntsteps, N_test, replace=False)
train_data = data.iloc[train_sel]
test_data = data.iloc[test_sel]
print(train_data)

# Training

start_time = time.time()
gpr = gpr.fit(train_data['Graphs'], train_data[target], repeat=1, verbose=True) # The training

end_time = time.time()
print("the total time consumption for " + str(N_train) + " steps is " + str(end_time - start_time) + "s.")

gpr.save(dir, filename='gpr_DFT_PotEng_3,2_'+str(N_train)+'.pkl') # Storing the model for future use

PY